# 08 — Silver Previous Application

**Credit Risk Intelligence Platform** — Camada Silver

Este notebook transforma a tabela Bronze `credit_risk.bronze.previous_application` em uma tabela Silver tratada, padronizada e preparada para análise e Machine Learning.

## Pipeline

```
credit_risk.bronze.previous_application  →  credit_risk.silver.previous_application
```

## Sobre a tabela previous_application

A tabela `previous_application` contém todas as aplicações anteriores de crédito do cliente no Home Credit. Cada registro representa uma aplicação anterior, identificada por `SK_ID_PREV`. O relacionamento com a tabela de aplicação atual é baseado em `SK_ID_CURR`.

A tabela Silver permanece no **nível original dos registros** — nenhuma agregação por cliente é realizada.

## Transformações aplicadas

1. **Remoção de metadados Bronze** — colunas `_ingestion_timestamp` e `_source_file`
2. **Padronização de categorias** — `trim()` em colunas string
3. **Tratamento de NULLs categóricos** — `NAME_TYPE_SUITE` e `PRODUCT_COMBINATION` → `'Unknown'`
4. **Anomalia DAYS_* = 365243** — conversão para NULL + flags de anomalia
5. **Flags de validação** — valores monetários negativos, SELLERPLACE_AREA inválido
6. **Colunas de controle** — timestamp, versão, origem, hash
7. **Auditoria** — registro completo da transformação

## Regras

> A Bronze **NÃO é modificada**. Todas as transformações criam novas tabelas Silver.
> Nenhum registro é removido sem justificativa documentada.
> Nenhuma agregação por cliente é realizada — a tabela permanece no nível de registros originais.
> Valores `XNA` (placeholder do Home Credit) são preservados — são valores originais do dataset.

In [0]:
# ============================================================================
# CÉLULA 1 — Configuração, Imports e Parâmetros
# ============================================================================
from pyspark.sql import functions as F, types as T, Window
from datetime import datetime, timezone
import uuid

# ----------------------------------------------------------------------------
# Parâmetros do pipeline
# ----------------------------------------------------------------------------
PIPELINE_VERSION = "silver_v1.0"
NOTEBOOK_NAME = "08_silver_previous_application"
EXECUTION_ID = str(uuid.uuid4())
BATCH_ID = f"silver_prev_app_{datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S')}"
EXECUTION_TIMESTAMP = datetime.now(timezone.utc)

# ----------------------------------------------------------------------------
# Tabelas de origem (Bronze) e destino (Silver)
# ----------------------------------------------------------------------------
BRONZE_TABLE = "credit_risk.bronze.previous_application"
SILVER_TABLE = "credit_risk.silver.previous_application"
AUDIT_TABLE = "credit_risk.silver.audit_transformation"

# Tabelas Silver de application (para integridade referencial)
SILVER_APP_TRAIN = "credit_risk.silver.application_train"
SILVER_APP_TEST = "credit_risk.silver.application_test"

# ----------------------------------------------------------------------------
# Colunas de metadados Bronze a remover na Silver
# ----------------------------------------------------------------------------
BRONZE_META_COLS = ["_ingestion_timestamp", "_source_file"]

# Valor especial do Home Credit para dias não aplicáveis
DAYS_ANOMALY_VALUE = 365243

# ----------------------------------------------------------------------------
# Criar schema Silver se não existir
# ----------------------------------------------------------------------------
spark.sql("CREATE SCHEMA IF NOT EXISTS credit_risk.silver")
print(f"Schema credit_risk.silver verificado/criado.")

# ----------------------------------------------------------------------------
# Dicionário para registrar transformações aplicadas (para auditoria)
# ----------------------------------------------------------------------------
TRANSFORMATION_LOG = []

def log_transform(table_name, step, description, records_affected=0):
    """Registra uma transformação aplicada para auditoria."""
    TRANSFORMATION_LOG.append({
        "table": table_name,
        "step": step,
        "description": description,
        "records_affected": records_affected,
    })

print(f"⏱️ Execution ID: {EXECUTION_ID}")
print(f"📦 Batch ID: {BATCH_ID}")
print(f"🔧 Pipeline Version: {PIPELINE_VERSION}")

In [0]:
# ============================================================================
# CÉLULA 2 — Leitura da Bronze e Inspeção do Schema
# ============================================================================
# Carrega o DataFrame Bronze (sem modificá-lo) e inspeciona o schema real.

df_prev_app_bronze = spark.table(BRONZE_TABLE)

# Métricas básicas
bronze_row_count = df_prev_app_bronze.count()
bronze_col_count = len(df_prev_app_bronze.columns)

print("=" * 70)
print("INSPEÇÃO INICIAL — BRONZE")
print("=" * 70)
print(f"\n📊 {BRONZE_TABLE}")
print(f"   Registros: {bronze_row_count:,}")
print(f"   Colunas: {bronze_col_count}")

# ----------------------------------------------------------------------------
# Schema detalhado (tipos e nullable)
# ----------------------------------------------------------------------------
sep = "─" * 70
print(f"\n{sep}")
print("SCHEMA — previous_application (tipos e nullable)")
print(sep)
for field in df_prev_app_bronze.schema.fields:
    print(f"   {field.name:<35} {field.dataType.simpleString():<12} nullable={field.nullable}")

# ----------------------------------------------------------------------------
# Verificar colunas-chave esperadas
# ----------------------------------------------------------------------------
print(f"\n{sep}")
print("COLUNAS-CHAVE")
print(sep)
for c in ["SK_ID_PREV", "SK_ID_CURR"]:
    if c in df_prev_app_bronze.columns:
        print(f"   ✅ {c}: presente")
    else:
        print(f"   ❌ {c}: AUSENTE")

print("\n✅ Leitura da Bronze concluída!")

In [0]:
# ============================================================================
# CÉLULA 3 — Data Quality Inicial (Bronze)
# ============================================================================
# Análise de completude (NULLs), valores distintos e estatísticas básicas.

sep = "─" * 70

# ----------------------------------------------------------------------------
# NULLs por coluna
# ----------------------------------------------------------------------------
null_exprs = [F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)).alias(c) for c in df_prev_app_bronze.columns]
null_row = df_prev_app_bronze.agg(*null_exprs).collect()[0]
null_pairs = [(c, null_row[c]) for c in df_prev_app_bronze.columns if null_row[c] and null_row[c] > 0]
null_pairs.sort(key=lambda x: x[1], reverse=True)

print(sep)
print(f"NULLs POR COLUNA — {BRONZE_TABLE} ({len(null_pairs)} cols com nulls)")
print(sep)
for c, n in null_pairs:
    pct = n / bronze_row_count * 100
    print(f"   {c:<35} {n:>10,}  ({pct:.2f}%)")

# ----------------------------------------------------------------------------
# Valores distintos por coluna (apenas colunas-chave e categóricas)
# ----------------------------------------------------------------------------
print(f"\n{sep}")
print("VALORES DISTINCTOS — COLUNAS PRINCIPAIS")
print(sep)
key_distinct_cols = [
    "SK_ID_PREV", "SK_ID_CURR", "NAME_CONTRACT_TYPE", "NAME_CONTRACT_STATUS",
    "NAME_CLIENT_TYPE", "NAME_PORTFOLIO", "NAME_PRODUCT_TYPE",
    "NAME_YIELD_GROUP", "CHANNEL_TYPE", "CODE_REJECT_REASON",
    "FLAG_LAST_APPL_PER_CONTRACT", "NAME_PAYMENT_TYPE"
]
for c in key_distinct_cols:
    if c in df_prev_app_bronze.columns:
        d = df_prev_app_bronze.select(c).distinct().count()
        print(f"   {c:<35} {d:>10,}")

# ----------------------------------------------------------------------------
# Estatísticas numéricas (colunas-chave)
# ----------------------------------------------------------------------------
numeric_inspect = [
    "AMT_ANNUITY", "AMT_APPLICATION", "AMT_CREDIT", "AMT_DOWN_PAYMENT",
    "AMT_GOODS_PRICE", "DAYS_DECISION", "SELLERPLACE_AREA", "CNT_PAYMENT",
    "RATE_DOWN_PAYMENT"
]
print(f"\n{sep}")
print("ESTATÍSTICAS NUMÉRICAS (min, max, mean)")
print(sep)
for c in numeric_inspect:
    if c in df_prev_app_bronze.columns:
        stats = df_prev_app_bronze.select(c).summary("min", "max", "mean").collect()
        mean_val = round(float(stats[2][c]), 2) if stats[2][c] else "N/A"
        print(f"   {c:<30} min={str(stats[0][c]):>15}  max={str(stats[1][c]):>15}  mean={str(mean_val):>15}")

print("\n✅ Data Quality inicial concluída!")

In [0]:
# ============================================================================
# CÉLULA 4 — Validação de Identificadores e Duplicidades
# ============================================================================
# Valida chaves SK_ID_PREV e SK_ID_CURR, analisa duplicidades e distribuição.

sep = "─" * 70
print("=" * 70)
print("VALIDAÇÃO DE IDENTIFICADORES E DUPLICIDADES")
print("=" * 70)

# ----------------------------------------------------------------------------
# SK_ID_PREV
# ----------------------------------------------------------------------------
sk_prev_null = df_prev_app_bronze.filter(F.col("SK_ID_PREV").isNull()).count()
sk_prev_distinct = df_prev_app_bronze.select("SK_ID_PREV").distinct().count()
sk_prev_dups = bronze_row_count - sk_prev_distinct
print(f"\n   SK_ID_PREV:")
print(f"      NULL: {sk_prev_null}")
print(f"      Distinct: {sk_prev_distinct:,}")
print(f"      Duplicatas: {sk_prev_dups}")

# ----------------------------------------------------------------------------
# SK_ID_CURR
# ----------------------------------------------------------------------------
sk_curr_null = df_prev_app_bronze.filter(F.col("SK_ID_CURR").isNull()).count()
sk_curr_distinct = df_prev_app_bronze.select("SK_ID_CURR").distinct().count()
print(f"\n   SK_ID_CURR:")
print(f"      NULL: {sk_curr_null}")
print(f"      Distinct: {sk_curr_distinct:,}")

# ----------------------------------------------------------------------------
# Duplicidades
# ----------------------------------------------------------------------------
print(f"\n{sep}")
print("ANÁLISE DE DUPLICIDADES")
print(sep)

# Duplicidade completa
full_dups = bronze_row_count - df_prev_app_bronze.dropDuplicates().count()
print(f"\n   Duplicidade completa: {full_dups} linhas totalmente duplicadas")

# Duplicidade por SK_ID_PREV
print(f"   Duplicidade por SK_ID_PREV: {sk_prev_dups}")
if sk_prev_dups == 0:
    print("   → SK_ID_PREV é único — nenhum tratamento necessário")

# Duplicidade por chave composta
composite_dups = bronze_row_count - df_prev_app_bronze.select("SK_ID_CURR", "SK_ID_PREV").distinct().count()
print(f"   Duplicidade por (SK_ID_CURR + SK_ID_PREV): {composite_dups}")

# ----------------------------------------------------------------------------
# Distribuição de registros por SK_ID_CURR
# ----------------------------------------------------------------------------
print(f"\n{sep}")
print("DISTRIBUIÇÃO DE REGISTROS POR SK_ID_CURR")
print(sep)

per_curr_stats = df_prev_app_bronze.groupBy("SK_ID_CURR").count().select("count")
summary = per_curr_stats.summary("min", "max", "mean", "50%").collect()
for s in summary:
    print(f"   {s['summary']:<10}: {s['count']}")

# Distribuição por faixas
print(f"\n   Distribuição por faixas:")
bins = [(1, 1), (2, 5), (6, 10), (11, 20), (21, 50), (51, 100)]
for lo, hi in bins:
    cnt = df_prev_app_bronze.groupBy("SK_ID_CURR").count().filter(
        (F.col("count") >= lo) & (F.col("count") <= hi)
    ).count()
    print(f"      {lo:>3}-{hi:<3} registros: {cnt:>8,} clientes")

print("\n✅ Validação de identificadores e duplicidades concluída!")

In [0]:
# ============================================================================
# CÉLULA 5 — Análise de Colunas Categóricas
# ============================================================================
# Inspeciona as categorias reais das colunas categóricas principais.
# Verifica espaços, capitalização, valores vazios, NULLs e valores inesperados.

sep = "─" * 70
print("=" * 70)
print("ANÁLISE DE COLUNAS CATEGÓRICAS")
print("=" * 70)

cat_cols_inspect = [
    "NAME_CONTRACT_TYPE", "NAME_CONTRACT_STATUS", "NAME_CLIENT_TYPE",
    "NAME_PORTFOLIO", "NAME_PRODUCT_TYPE", "NAME_YIELD_GROUP",
    "CHANNEL_TYPE", "CODE_REJECT_REASON", "NAME_PAYMENT_TYPE",
    "NAME_SELLER_INDUSTRY", "FLAG_LAST_APPL_PER_CONTRACT",
    "NAME_CASH_LOAN_PURPOSE", "NAME_GOODS_CATEGORY", "NAME_TYPE_SUITE",
    "WEEKDAY_APPR_PROCESS_START", "PRODUCT_COMBINATION"
]

for c in cat_cols_inspect:
    if c in df_prev_app_bronze.columns:
        vals = df_prev_app_bronze.groupBy(c).count().orderBy(F.desc("count")).collect()
        null_cnt = df_prev_app_bronze.filter(F.col(c).isNull()).count()
        print(f"\n   {c} ({len(vals)} distinct, NULL={null_cnt:,}):")
        for r in vals[:10]:
            print(f"      {str(r[c]):<40} {r['count']:>12,}")
        if len(vals) > 10:
            print(f"      ... e mais {len(vals) - 10} valores")

print("\n✅ Análise de colunas categóricas concluída!")

In [0]:
# ============================================================================
# CÉLULA 6 — Análise de Valores Monetários
# ============================================================================
# Inspeciona colunas monetárias: negativos, zeros, NULLs, valores extremos.

sep = "─" * 70
print("=" * 70)
print("ANÁLISE DE VALORES MONETÁRIOS")
print("=" * 70)

amt_cols = [c for c in df_prev_app_bronze.columns
            if c.startswith("AMT_") and c in [
                "AMT_ANNUITY", "AMT_APPLICATION", "AMT_CREDIT",
                "AMT_DOWN_PAYMENT", "AMT_GOODS_PRICE"
            ]]

for c in amt_cols:
    null_cnt = df_prev_app_bronze.filter(F.col(c).isNull()).count()
    neg_cnt = df_prev_app_bronze.filter(F.col(c) < 0).count()
    zero_cnt = df_prev_app_bronze.filter(F.col(c) == 0).count()
    stats = df_prev_app_bronze.filter(F.col(c).isNotNull()).select(
        F.min(c).alias("min"), F.max(c).alias("max")).collect()[0]
    print(f"\n   {c}:")
    print(f"      NULL: {null_cnt:,} ({null_cnt/bronze_row_count*100:.2f}%)")
    print(f"      Negativos: {neg_cnt:,}")
    print(f"      Zeros: {zero_cnt:,} ({zero_cnt/bronze_row_count*100:.2f}%)")
    print(f"      Min: {stats['min']:.2f}  Max: {stats['max']:.2f}")
    if neg_cnt > 0:
        print(f"      ⚠️ WARNING: {neg_cnt} valores negativos — preservados, podem ter significado próprio")

# RATE columns
print(f"\n{sep}")
print("COLUNAS DE TAXA (RATE_*)")
print(sep)
rate_cols = ["RATE_DOWN_PAYMENT", "RATE_INTEREST_PRIMARY", "RATE_INTEREST_PRIVILEGED"]
for c in rate_cols:
    if c in df_prev_app_bronze.columns:
        null_cnt = df_prev_app_bronze.filter(F.col(c).isNull()).count()
        neg_cnt = df_prev_app_bronze.filter(F.col(c) < 0).count()
        print(f"   {c:<30} NULL: {null_cnt:,} ({null_cnt/bronze_row_count*100:.2f}%)  Negativos: {neg_cnt}")
        if null_cnt / bronze_row_count > 0.9:
            print(f"      ⚠️ Quase todos os valores são NULL — coluna quase vazia")

print("\n✅ Análise de valores monetários concluída!")

In [0]:
# ============================================================================
# CÉLULA 7 — Análise de Campos Temporais (DAYS_*)
# ============================================================================
# Inspeciona todas as colunas DAYS_* e identifica valores especiais.
# O valor 365243 é usado pelo Home Credit para indicar "não aplicável".

sep = "─" * 70
print("=" * 70)
print("ANÁLISE DE CAMPOS TEMPORAIS (DAYS_*)")
print("=" * 70)

days_cols = [c for c in df_prev_app_bronze.columns if c.startswith("DAYS_")]
print(f"\nColunas DAYS_* encontradas: {days_cols}")

for c in days_cols:
    null_cnt = df_prev_app_bronze.filter(F.col(c).isNull()).count()
    stats = df_prev_app_bronze.filter(F.col(c).isNotNull()).select(
        F.min(c).alias("min"), F.max(c).alias("max")).collect()[0]
    anomaly_cnt = df_prev_app_bronze.filter(F.col(c) == DAYS_ANOMALY_VALUE).count()

    print(f"\n   {c}:")
    print(f"      NULL: {null_cnt:,} ({null_cnt/bronze_row_count*100:.2f}%)")
    print(f"      Min: {stats['min']}  Max: {stats['max']}")
    print(f"      Valor 365243: {anomaly_cnt:,} ({anomaly_cnt/bronze_row_count*100:.2f}%)")

    if anomaly_cnt > 0:
        print(f"      → Anomalia 365243 detectada — flag FLAG_{c}_ANOMALY sera criada")
        print(f"        O valor 365243 representa 'nao aplicavel' no Home Credit")
        print(f"        Tratamento: converter para NULL + criar flag de anomalia")

# ----------------------------------------------------------------------------
# SELLERPLACE_AREA
# ----------------------------------------------------------------------------
print(f"\n{sep}")
print("SELLERPLACE_AREA")
print(sep)
spa_neg = df_prev_app_bronze.filter(F.col("SELLERPLACE_AREA") < 0).count()
spa_stats = df_prev_app_bronze.select("SELLERPLACE_AREA").summary("min", "max", "mean").collect()
print(f"   Min: {spa_stats[0]['SELLERPLACE_AREA']}, Max: {spa_stats[1]['SELLERPLACE_AREA']}, Mean: {float(spa_stats[2]['SELLERPLACE_AREA']):.1f}")
print(f"   Valores negativos: {spa_neg:,} ({spa_neg/bronze_row_count*100:.2f}%)")
if spa_neg > 0:
    print(f"   → Valores negativos (-1) representam 'area desconhecida' — flag sera criada")

# ----------------------------------------------------------------------------
# NFLAG_INSURED_ON_APPROVAL
# ----------------------------------------------------------------------------
print(f"\n{sep}")
print("NFLAG_INSURED_ON_APPROVAL")
print(sep)
nflag_dist = df_prev_app_bronze.groupBy("NFLAG_INSURED_ON_APPROVAL").count().orderBy("NFLAG_INSURED_ON_APPROVAL").collect()
for r in nflag_dist:
    print(f"   {r['NFLAG_INSURED_ON_APPROVAL']}: {r['count']:,}")

print("\n✅ Análise de campos temporais concluída!")

In [0]:
# ============================================================================
# CÉLULA 8 — Integridade Referencial
# ============================================================================
# Valida a relação SK_ID_CURR da previous_application contra as tabelas Silver
# de application_train e application_test.

sep = "─" * 70
print("=" * 70)
print("INTEGRIDADE REFERENCIAL — previous_application vs application")
print("=" * 70)

prev_sk = df_prev_app_bronze.select("SK_ID_CURR").distinct()
prev_sk_count = prev_sk.count()

# ----------------------------------------------------------------------------
# Verificar se tabelas Silver de application existem
# ----------------------------------------------------------------------------
app_tables = {}
for name, tbl in [("application_train", SILVER_APP_TRAIN), ("application_test", SILVER_APP_TEST)]:
    try:
        df_app = spark.table(tbl)
        app_tables[name] = df_app.select("SK_ID_CURR").distinct()
        print(f"   ✅ {tbl}: {app_tables[name].count():,} SK_ID_CURR distintos")
    except Exception:
        print(f"   ⚠️  {tbl}: tabela não encontrada — pulando")

# ----------------------------------------------------------------------------
# Integridade vs application_train
# ----------------------------------------------------------------------------
if "application_train" in app_tables:
    matched_train = prev_sk.join(app_tables["application_train"], "SK_ID_CURR", "inner").count()
    unmatched_train = prev_sk_count - matched_train
    print(f"\n📊 previous_application SK_ID_CURR vs application_train:")
    print(f"   Correspondidos: {matched_train:,} ({matched_train/prev_sk_count*100:.2f}%)")
    print(f"   Sem correspondência: {unmatched_train:,} ({unmatched_train/prev_sk_count*100:.2f}%)")

# ----------------------------------------------------------------------------
# Integridade vs application_test
# ----------------------------------------------------------------------------
if "application_test" in app_tables:
    matched_test = prev_sk.join(app_tables["application_test"], "SK_ID_CURR", "inner").count()
    unmatched_test = prev_sk_count - matched_test
    print(f"\n📊 previous_application SK_ID_CURR vs application_test:")
    print(f"   Correspondidos: {matched_test:,} ({matched_test/prev_sk_count*100:.2f}%)")
    print(f"   Sem correspondência: {unmatched_test:,} ({unmatched_test/prev_sk_count*100:.2f}%)")

# ----------------------------------------------------------------------------
# Integridade vs application combinado
# ----------------------------------------------------------------------------
if len(app_tables) == 2:
    all_app_sk = app_tables["application_train"].union(app_tables["application_test"]).distinct()
    all_app_count = all_app_sk.count()
    matched_all = prev_sk.join(all_app_sk, "SK_ID_CURR", "inner").count()
    unmatched_all = prev_sk_count - matched_all
    print(f"\n📊 previous_application SK_ID_CURR vs application (train + test combinado):")
    print(f"   Total app SK_ID_CURR: {all_app_count:,}")
    print(f"   Correspondidos: {matched_all:,} ({matched_all/prev_sk_count*100:.2f}%)")
    print(f"   Sem correspondência: {unmatched_all:,} ({unmatched_all/prev_sk_count*100:.2f}%)")
    if unmatched_all == 0:
        print(f"   → ✅ Todos os SK_ID_CURR têm correspondência em application")
    else:
        print(f"   → ⚠️ {unmatched_all} SK_ID_CURR sem correspondência (registros preservados)")

print("\n✅ Integridade referencial validada!")

In [0]:
# ============================================================================
# CÉLULA 9 — Funções de Transformação Reutilizáveis
# ============================================================================
# Funções modulares aplicadas na transformação Bronze → Silver.

def remove_bronze_metadata(df, table_name):
    """Remove colunas de metadados da Bronze."""
    cols_to_drop = [c for c in BRONZE_META_COLS if c in df.columns]
    if cols_to_drop:
        df = df.drop(*cols_to_drop)
        log_transform(table_name, "remove_metadata", f"Removidas colunas Bronze: {cols_to_drop}")
    return df


def standardize_categories(df, table_name):
    """Padroniza colunas categóricas: trim de espaços extras."""
    string_cols = [f.name for f in df.schema.fields if f.dataType.simpleString() == "string"]
    for col_name in string_cols:
        df = df.withColumn(col_name, F.trim(F.col(col_name)))
    log_transform(table_name, "standardize_categories",
                  f"Trim aplicado em {len(string_cols)} colunas string")
    print(f"   ✅ Padronização: trim aplicado em {len(string_cols)} colunas string")
    return df


def treat_nulls_categorical(df, table_name, row_count):
    """Trata NULLs em colunas categóricas substituindo por 'Unknown'.
    Apenas colunas com NULL significativo são tratadas."""
    # NAME_TYPE_SUITE: 49.12% NULL → 'Unknown'
    if "NAME_TYPE_SUITE" in df.columns:
        null_cnt = df.filter(F.col("NAME_TYPE_SUITE").isNull()).count()
        if null_cnt > 0:
            df = df.withColumn("NAME_TYPE_SUITE",
                F.when(F.col("NAME_TYPE_SUITE").isNull(), "Unknown").otherwise(F.col("NAME_TYPE_SUITE")))
            log_transform(table_name, "null_categorical",
                f"NAME_TYPE_SUITE: NULL → 'Unknown' ({null_cnt} registros)", null_cnt)
            print(f"   ✅ NAME_TYPE_SUITE: {null_cnt} NULLs → 'Unknown'")

    # PRODUCT_COMBINATION: 0.02% NULL → 'Unknown'
    if "PRODUCT_COMBINATION" in df.columns:
        null_cnt = df.filter(F.col("PRODUCT_COMBINATION").isNull()).count()
        if null_cnt > 0:
            df = df.withColumn("PRODUCT_COMBINATION",
                F.when(F.col("PRODUCT_COMBINATION").isNull(), "Unknown").otherwise(F.col("PRODUCT_COMBINATION")))
            log_transform(table_name, "null_categorical",
                f"PRODUCT_COMBINATION: NULL → 'Unknown' ({null_cnt} registros)", null_cnt)
            print(f"   ✅ PRODUCT_COMBINATION: {null_cnt} NULLs → 'Unknown'")
    return df


def treat_days_anomaly(df, table_name, row_count):
    """Trata o valor especial 365243 nas colunas DAYS_*.
    Regra: converter 365243 para NULL e criar flag de anomalia.
    O valor 365243 representa 'não aplicável' no dataset Home Credit."""
    days_cols = [c for c in df.columns if c.startswith("DAYS_") and c != "DAYS_DECISION"]
    for c in days_cols:
        anomaly_cnt = df.filter(F.col(c) == DAYS_ANOMALY_VALUE).count()
        if anomaly_cnt > 0:
            flag_name = f"FLAG_{c}_ANOMALY"
            df = df.withColumn(flag_name,
                F.when(F.col(c) == DAYS_ANOMALY_VALUE, 1).otherwise(0))
            df = df.withColumn(c,
                F.when(F.col(c) == DAYS_ANOMALY_VALUE, None).otherwise(F.col(c)))
            log_transform(table_name, "days_anomaly",
                f"{c}=365243 → NULL + {flag_name}=1 ({anomaly_cnt} registros)", anomaly_cnt)
            print(f"   ✅ {c}: {anomaly_cnt} registros (365243 → NULL + flag)")
    return df


def add_validation_flags(df, table_name, row_count):
    """Adiciona flags de validação para valores potencialmente inválidos."""
    flags_added = []

    # AMT_DOWN_PAYMENT negativo
    if "AMT_DOWN_PAYMENT" in df.columns:
        neg = df.filter(F.col("AMT_DOWN_PAYMENT") < 0).count()
        df = df.withColumn("FLAG_AMT_DOWN_PAYMENT_NEGATIVE",
            F.when(F.col("AMT_DOWN_PAYMENT") < 0, 1).otherwise(0))
        flags_added.append(("FLAG_AMT_DOWN_PAYMENT_NEGATIVE", neg))

    # SELLERPLACE_AREA negativo (-1 = area desconhecida)
    if "SELLERPLACE_AREA" in df.columns:
        neg = df.filter(F.col("SELLERPLACE_AREA") < 0).count()
        df = df.withColumn("FLAG_SELLERPLACE_AREA_INVALID",
            F.when(F.col("SELLERPLACE_AREA") < 0, 1).otherwise(0))
        flags_added.append(("FLAG_SELLERPLACE_AREA_INVALID", neg))

    # AMT_CREDIT NULL (apenas 1 registro)
    if "AMT_CREDIT" in df.columns:
        null_cnt = df.filter(F.col("AMT_CREDIT").isNull()).count()
        df = df.withColumn("FLAG_AMT_CREDIT_NULL",
            F.when(F.col("AMT_CREDIT").isNull(), 1).otherwise(0))
        flags_added.append(("FLAG_AMT_CREDIT_NULL", null_cnt))

    for flag_name, affected_count in flags_added:
        log_transform(table_name, "validation_flag",
            f"{flag_name}: {affected_count} registros marcados", affected_count)
        print(f"   ✅ {flag_name}: {affected_count} registros")
    return df


def add_control_columns(df, source_table):
    """Adiciona colunas de controle técnicas da Silver."""
    df = df.withColumn("silver_processing_timestamp", F.current_timestamp())
    df = df.withColumn("silver_processing_date", F.current_date())
    df = df.withColumn("silver_pipeline_version", F.lit(PIPELINE_VERSION))
    df = df.withColumn("source_table", F.lit(source_table))

    # record_hash: hash MD5 de todas as colunas de dados para rastreabilidade
    data_cols = [c for c in df.columns if c not in [
        "silver_processing_timestamp", "silver_processing_date",
        "silver_pipeline_version", "source_table"
    ]]
    hash_expr = F.concat_ws("||", *[F.coalesce(F.col(c).cast("string"), F.lit("NULL")) for c in data_cols])
    df = df.withColumn("record_hash", F.md5(hash_expr))
    print(f"   ✅ Colunas de controle adicionadas (timestamp, date, version, source, hash)")
    return df


def apply_silver_transformations(df, table_name, source_table, row_count):
    """Aplica todas as transformações Silver em sequência."""
    print(f"\n{'─' * 60}")
    print(f"🔧 Transformando: {table_name}")
    print(f"{'─' * 60}")

    df = remove_bronze_metadata(df, table_name)
    df = standardize_categories(df, table_name)
    df = treat_nulls_categorical(df, table_name, row_count)
    df = treat_days_anomaly(df, table_name, row_count)
    df = add_validation_flags(df, table_name, row_count)
    df = add_control_columns(df, source_table)

    print(f"   ✅ Transformações concluídas para {table_name}")
    return df


print("✅ Funções de transformação definidas!")

In [0]:
# ============================================================================
# CÉLULA 10 — Execução das Transformações
# ============================================================================
# Aplica as transformações Silver na tabela previous_application.
# O DataFrame Bronze original não é modificado.

EXEC_START = datetime.now(timezone.utc)

print("=" * 70)
print("TRANSFORMAÇÃO SILVER — previous_application")
print("=" * 70)
transform_start = datetime.now(timezone.utc)

df_prev_app_silver = apply_silver_transformations(
    df_prev_app_bronze, SILVER_TABLE, BRONZE_TABLE, bronze_row_count
)

transform_end = datetime.now(timezone.utc)
transform_duration = (transform_end - transform_start).total_seconds()
silver_row_count = df_prev_app_silver.count()
silver_col_count = len(df_prev_app_silver.columns)

print(f"\n   Bronze: {bronze_row_count:,} rows x {bronze_col_count} cols")
print(f"   Silver: {silver_row_count:,} rows x {silver_col_count} cols")
print(f"   Duração: {transform_duration:.1f}s")

print(f"\n{'=' * 70}")
print(f"⏱️ Tempo total de transformação: {transform_duration:.1f}s")
print(f"{'=' * 70}")

In [0]:
# ============================================================================
# CÉLULA 11 — Escrita da Tabela Silver (Delta Lake)
# ============================================================================
# Grava a tabela Silver usando mode("overwrite") com overwriteSchema.

print("=" * 70)
print("GRAVAÇÃO DA TABELA SILVER")
print("=" * 70)

print(f"\n📊 Gravando {SILVER_TABLE}...")
write_start = datetime.now(timezone.utc)

df_prev_app_silver.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .format("delta") \
    .saveAsTable(SILVER_TABLE)

write_end = datetime.now(timezone.utc)
write_duration = (write_end - write_start).total_seconds()
print(f"   ✅ {SILVER_TABLE} gravada em {write_duration:.1f}s")
print(f"      Registros: {silver_row_count:,} | Colunas: {silver_col_count}")

EXEC_END = datetime.now(timezone.utc)
TOTAL_DURATION = (EXEC_END - EXEC_START).total_seconds()

print(f"\n{'=' * 70}")
print("✅ TABELA SILVER GRAVADA COM SUCESSO!")
print(f"{'=' * 70}")

In [0]:
# ============================================================================
# CÉLULA 12 — Auditoria da Transformação
# ============================================================================
# Cria/atualiza a tabela credit_risk.silver.audit_transformation (append mode).
from pyspark.sql.types import (StructType, StructField, StringType,
    IntegerType, DoubleType, TimestampType, LongType)

audit_record = {
    "execution_timestamp": EXECUTION_TIMESTAMP,
    "execution_id": EXECUTION_ID,
    "batch_id": BATCH_ID,
    "source_table": BRONZE_TABLE,
    "target_table": SILVER_TABLE,
    "source_row_count": bronze_row_count,
    "target_row_count": silver_row_count,
    "records_inserted": silver_row_count,
    "records_removed": 0,
    "records_changed": silver_row_count,
    "processing_duration_seconds": float(TOTAL_DURATION),
    "pipeline_version": PIPELINE_VERSION,
    "execution_status": "SUCCESS",
    "error_message": "",
}

audit_schema = StructType([
    StructField("execution_timestamp", TimestampType(), True),
    StructField("execution_id", StringType(), True),
    StructField("batch_id", StringType(), True),
    StructField("source_table", StringType(), True),
    StructField("target_table", StringType(), True),
    StructField("source_row_count", LongType(), True),
    StructField("target_row_count", LongType(), True),
    StructField("records_inserted", LongType(), True),
    StructField("records_removed", IntegerType(), True),
    StructField("records_changed", LongType(), True),
    StructField("processing_duration_seconds", DoubleType(), True),
    StructField("pipeline_version", StringType(), True),
    StructField("execution_status", StringType(), True),
    StructField("error_message", StringType(), True),
])

audit_df = spark.createDataFrame([audit_record], schema=audit_schema)

print(f"📊 Persistindo auditoria em {AUDIT_TABLE}...")
audit_df.write.mode("append").format("delta").saveAsTable(AUDIT_TABLE)

print(f"✅ Auditoria registrada: 1 registro em {AUDIT_TABLE}")
print("\nRegistros de auditoria (últimos 10):")
display(spark.table(AUDIT_TABLE).orderBy(F.col("execution_timestamp").desc()).limit(10))

In [0]:
# ============================================================================
# CÉLULA 13 — Data Quality Pós-Transformação (Bronze vs Silver)
# ============================================================================
# Compara métricas de qualidade antes (Bronze) e depois (Silver).

def compute_dq_metrics(df, table_name):
    """Computa métricas de DQ: row_count, col_count, null_count, duplicate_count."""
    row_count = df.count()
    col_count = len(df.columns)
    null_exprs = [F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)) for c in df.columns]
    total_nulls = df.agg(*null_exprs).collect()[0]
    null_sum = sum([total_nulls[i] for i in range(len(df.columns))])
    if "SK_ID_PREV" in df.columns:
        dup_count = row_count - df.select("SK_ID_PREV").distinct().count()
    else:
        dup_count = 0
    return {
        "table": table_name,
        "row_count": row_count,
        "col_count": col_count,
        "null_count": null_sum,
        "null_percentage": round(null_sum / (row_count * col_count) * 100, 2) if row_count > 0 else 0,
        "duplicate_count": dup_count,
    }

# ----------------------------------------------------------------------------
# Ler tabela Silver recém-criada
# ----------------------------------------------------------------------------
df_silver = spark.table(SILVER_TABLE)

# ----------------------------------------------------------------------------
# Métricas Bronze vs Silver
# ----------------------------------------------------------------------------
sep = "─" * 75
print("=" * 70)
print("DATA QUALITY: BRONZE vs SILVER")
print("=" * 70)

bronze_m = compute_dq_metrics(df_prev_app_bronze, BRONZE_TABLE)
silver_m = compute_dq_metrics(df_silver, SILVER_TABLE)

print(f"\n📊 previous_application")
print(f"{'Métrica':<30} {'Bronze':>15} {'Silver':>15} {'Delta':>15}")
print(sep)
for key in ["row_count", "col_count", "null_count", "null_percentage", "duplicate_count"]:
    b = bronze_m[key]
    s = silver_m[key]
    d = s - b
    print(f"{key:<30} {b:>15,} {s:>15,} {d:>+15,}")

# ----------------------------------------------------------------------------
# Verificações específicas
# ----------------------------------------------------------------------------
print(f"\n{sep}")
print("VERIFICAÇÕES ESPECÍFICAS")
print(sep)

# Colunas de controle presentes
control_cols = ["silver_processing_timestamp", "silver_processing_date",
                "silver_pipeline_version", "source_table", "record_hash"]
for c in control_cols:
    present = c in df_silver.columns
    print(f"   Coluna {c}: {'✅ presente' if present else '❌ ausente'}")

# Colunas Bronze removidas
print(f"\n   Colunas Bronze removidas:")
for c in BRONZE_META_COLS:
    present = c in df_silver.columns
    print(f"      {c}: {'❌ ainda presente' if present else '✅ removida'}")

# Flags de anomalia DAYS_*
anomaly_flags = [c for c in df_silver.columns if c.startswith("FLAG_DAYS_") and c.endswith("_ANOMALY")]
print(f"\n   Flags de anomalia DAYS_* ({len(anomaly_flags)}):")
for flag in anomaly_flags:
    count = df_silver.filter(F.col(flag) == 1).count()
    print(f"      {flag}: {count} registros")

# Flags de validação
validation_flags = [c for c in df_silver.columns
    if c.startswith("FLAG_") and not c.endswith("_ANOMALY")
    and c != "FLAG_LAST_APPL_PER_CONTRACT"]
print(f"\n   Flags de validação ({len(validation_flags)}):")
for flag in validation_flags:
    count = df_silver.filter(F.col(flag) == 1).count()
    print(f"      {flag}: {count} registros")

# SK_ID_PREV uniqueness na Silver
silver_sk_dups = silver_row_count - df_silver.select("SK_ID_PREV").distinct().count()
print(f"\n   SK_ID_PREV duplicatas na Silver: {silver_sk_dups} (esperado: 0)")

# NAME_CONTRACT_STATUS distribution preservada
print(f"\n   NAME_CONTRACT_STATUS (Silver):")
status_dist = df_silver.groupBy("NAME_CONTRACT_STATUS").count().orderBy(F.desc("count")).collect()
for r in status_dist:
    print(f"      {r['NAME_CONTRACT_STATUS']:<20} {r['count']:>12,}")

print("\n✅ Data Quality pós-transformação concluída!")

In [0]:
# ============================================================================
# CÉLULA 14 — Validação Final e Amostras
# ============================================================================
# Valida que a tabela Silver está correta e coerente com a Bronze.

sep = "─" * 70
print("=" * 70)
print("VALIDAÇÃO FINAL — TABELA SILVER")
print("=" * 70)

print(f"\n📊 {SILVER_TABLE}")
print(sep)

# Comparar row count
assert silver_row_count == bronze_row_count, \
    f"Row count mismatch: Bronze={bronze_row_count} vs Silver={silver_row_count}"
print(f"   ✅ Row count: {silver_row_count:,} (igual à Bronze)")

# Comparar SK_ID_PREV uniqueness
silver_dups = silver_row_count - df_silver.select("SK_ID_PREV").distinct().count()
assert silver_dups == 0, f"Duplicatas encontradas: {silver_dups}"
print(f"   ✅ SK_ID_PREV: único (0 duplicatas)")

# Colunas Silver vs Bronze
print(f"   Colunas Bronze: {bronze_col_count}")
print(f"   Colunas Silver: {silver_col_count}")
print(f"   Colunas adicionadas: {silver_col_count - bronze_col_count}")
print(f"     - Removidas: 2 (metadados Bronze)")
print(f"     - Adicionadas: 5 flags DAYS_* + 3 flags validação + 5 colunas controle")

# ----------------------------------------------------------------------------
# Amostra
# ----------------------------------------------------------------------------
print(f"\n{sep}")
print(f"AMOSTRA — {SILVER_TABLE} (primeiras 20 linhas)")
print(sep)

sample_cols = [
    "SK_ID_PREV", "SK_ID_CURR", "NAME_CONTRACT_TYPE", "NAME_CONTRACT_STATUS",
    "AMT_APPLICATION", "AMT_CREDIT", "AMT_ANNUITY",
    "DAYS_DECISION", "DAYS_FIRST_DRAWING", "DAYS_LAST_DUE",
    "NAME_CLIENT_TYPE", "NAME_PORTFOLIO",
    "FLAG_DAYS_FIRST_DRAWING_ANOMALY", "FLAG_DAYS_LAST_DUE_ANOMALY",
    "silver_processing_timestamp", "silver_pipeline_version",
    "source_table", "record_hash"
]
sample_cols = [c for c in sample_cols if c in df_silver.columns]
display(df_silver.select(*sample_cols).limit(20))

# ----------------------------------------------------------------------------
# Estatísticas resumidas
# ----------------------------------------------------------------------------
print(f"\n{sep}")
print("ESTATÍSTICAS RESUMIDAS (Silver)")
print(sep)

silver_sk_prev = df_silver.select("SK_ID_PREV").distinct().count()
silver_sk_curr = df_silver.select("SK_ID_CURR").distinct().count()
print(f"   SK_ID_PREV distintos: {silver_sk_prev:,}")
print(f"   SK_ID_CURR distintos: {silver_sk_curr:,}")
print(f"   Registros por cliente (média): {silver_row_count / silver_sk_curr:.2f}")

print("\n✅ Validação final concluída com sucesso!")

In [0]:
# ============================================================================
# CÉLULA 15 — Resumo Final da Execução
# ============================================================================
# Exibe um resumo completo da transformação.

print("=" * 60)
print("SILVER PREVIOUS_APPLICATION - RESUMO")
print("=" * 60)

print(f"\nOrigem:\n  {BRONZE_TABLE}")
print(f"\nDestino:\n  {SILVER_TABLE}")
print(f"\nRegistros Bronze:\n  {bronze_row_count:,}")
print(f"\nRegistros Silver:\n  {silver_row_count:,}")
print(f"\nRegistros removidos:\n  0")
print(f"\nRegistros alterados:\n  {silver_row_count:,} (transformações aplicadas)")
print(f"\nColunas:\n  Bronze: {bronze_col_count}")
print(f"  Silver: {silver_col_count}")

# NULLs tratados
nulls_treated = sum(t["records_affected"] for t in TRANSFORMATION_LOG if t["step"] == "null_categorical")
print(f"\nNULLs tratados (categóricos):\n  {nulls_treated:,}")

# Duplicidades
print(f"\nDuplicidades identificadas:\n  Completa: 0")
print(f"  SK_ID_PREV: 0")
print(f"\nDuplicidades removidas:\n  0 (não havia duplicidades)")

# Anomalias DAYS_*
anomaly_treated = sum(t["records_affected"] for t in TRANSFORMATION_LOG if t["step"] == "days_anomaly")
print(f"\nAnomalias DAYS_*=365243 tratadas:\n  {anomaly_treated:,} registros em 5 colunas")

# Valores inválidos identificados
print(f"\nValores inválidos identificados:")
for t in TRANSFORMATION_LOG:
    if t["step"] == "validation_flag":
        print(f"  • {t['description']}")

# Integridade referencial
print(f"\nIntegridade referencial com application:")
if len(app_tables) == 2:
    all_app_sk = app_tables["application_train"].union(app_tables["application_test"]).distinct()
    unmatched = prev_sk_count - prev_sk.join(all_app_sk, "SK_ID_CURR", "inner").count()
    print(f"  Sem correspondência: {unmatched} ({unmatched/prev_sk_count*100:.2f}%)")
else:
    print(f"  Verificação não realizada")

# Transformações aplicadas
print(f"\nRegras aplicadas ({len(TRANSFORMATION_LOG)}):")
for t in TRANSFORMATION_LOG:
    print(f"  • {t['step']}: {t['description']}")

# Warnings
warnings = [t for t in TRANSFORMATION_LOG if "WARNING" in t.get("description", "")]
print(f"\nWarnings:\n  {len(warnings)}")

print(f"\nStatus:\n  SUCCESS")
print(f"\nTempo:\n  {TOTAL_DURATION:.1f} segundos")
print(f"\n⏱️ Execution ID: {EXECUTION_ID}")
print(f"📦 Batch ID: {BATCH_ID}")
print(f"🔧 Pipeline: {PIPELINE_VERSION}")
print(f"\n{'=' * 60}")
print("✅ PIPELINE SILVER PREVIOUS_APPLICATION CONCLUÍDO COM SUCESSO!")
print(f"{'=' * 60}")

## Transformações Aplicadas — Documentação

### 1. Remoção de metadados Bronze
Colunas `_ingestion_timestamp` e `_source_file` removidas (substituídas por colunas de controle Silver).

### 2. Padronização de categorias
- `trim()` aplicado em todas as colunas string para remover espaços extras
- Valores `XNA` (placeholder do Home Credit) foram **preservados** — são valores originais do dataset
- Nenhuma alteração semântica foi realizada

### 3. Tratamento de NULLs categóricos

| Coluna | NULLs | % | Tratamento |
|--------|-------|---|------------|
| `NAME_TYPE_SUITE` | 820.405 | 49,12% | NULL → `'Unknown'` |
| `PRODUCT_COMBINATION` | 346 | 0,02% | NULL → `'Unknown'` |

- Colunas numéricas com NULLs (AMT_ANNUITY, AMT_DOWN_PAYMENT, AMT_GOODS_PRICE, CNT_PAYMENT, RATE_*) foram **preservadas** — NULLs têm significado próprio (ex: sem entrada, sem anuidade)

### 4. Anomalia DAYS_* = 365243
O valor 365243 representa "não aplicável" no dataset Home Credit. Foi aplicado:
- Conversão para NULL
- Criação de flag de anomalia para cada coluna

| Coluna | Registros com 365243 | Flag criada |
|--------|---------------------|-------------|
| `DAYS_FIRST_DRAWING` | 934.444 | `FLAG_DAYS_FIRST_DRAWING_ANOMALY` |
| `DAYS_FIRST_DUE` | 40.645 | `FLAG_DAYS_FIRST_DUE_ANOMALY` |
| `DAYS_LAST_DUE_1ST_VERSION` | 93.864 | `FLAG_DAYS_LAST_DUE_1ST_VERSION_ANOMALY` |
| `DAYS_LAST_DUE` | 211.221 | `FLAG_DAYS_LAST_DUE_ANOMALY` |
| `DAYS_TERMINATION` | 225.913 | `FLAG_DAYS_TERMINATION_ANOMALY` |

- `DAYS_DECISION` não possui o valor 365243 (min=-2922, max=-1) — não foi tratado

### 5. Flags de validação

| Flag | Descrição | Registros |
|------|-----------|-----------|
| `FLAG_AMT_DOWN_PAYMENT_NEGATIVE` | AMT_DOWN_PAYMENT < 0 | 2 |
| `FLAG_SELLERPLACE_AREA_INVALID` | SELLERPLACE_AREA < 0 (-1 = desconhecido) | 762.675 |
| `FLAG_AMT_CREDIT_NULL` | AMT_CREDIT IS NULL | 1 |

- Valores marcados são **preservados** — flags permitem filtragem posterior sem perda de dados

### 6. Duplicidades
- 0 linhas totalmente duplicadas
- 0 duplicatas por SK_ID_PREV (chave primária única)
- 0 duplicatas por (SK_ID_CURR + SK_ID_PREV)
- Nenhuma deduplicação foi necessária

### 7. Integridade referencial
- previous_application.SK_ID_CURR → silver.application_train/test.SK_ID_CURR
- Nenhum registro foi removido por falta de correspondência

### 8. Colunas de controle Silver
| Coluna | Tipo | Descrição |
|--------|------|------------|
| `silver_processing_timestamp` | timestamp | Momento da transformação |
| `silver_processing_date` | date | Data da transformação |
| `silver_pipeline_version` | string | Versão do pipeline (`silver_v1.0`) |
| `source_table` | string | Tabela de origem Bronze |
| `record_hash` | string | Hash MD5 de todos os campos para rastreabilidade |

### 9. Schema da tabela Silver
- **Bronze**: 39 colunas (37 dados + 2 metadados)
- **Silver**: 45 colunas (37 dados + 5 flags + 5 controle - 2 metadados removidos)
- Nenhuma coluna original foi modificada em tipo ou semântica
- Apenas valores 365243 em DAYS_* foram convertidos para NULL (com flag)

### 10. Colunas quase vazias (WARNING)
- `RATE_INTEREST_PRIMARY`: 99,64% NULL — preservada mas quase sem informação útil
- `RATE_INTEREST_PRIVILEGED`: 99,64% NULL — preservada mas quase sem informação útil
- Estas colunas não foram removidas, mas seu uso deve ser cauteloso em análises futuras

### 11. Valores XNA
O valor `XNA` é usado pelo Home Credit como placeholder para "não aplicável" em colunas categóricas. Foi **preservado** como valor original:
- `NAME_CONTRACT_TYPE`: 346 XNA
- `NAME_CLIENT_TYPE`: 1.941 XNA
- `NAME_PORTFOLIO`: 372.230 XNA
- `NAME_PRODUCT_TYPE`: 1.063.666 XNA
- `NAME_YIELD_GROUP`: 517.215 XNA
- `NAME_SELLER_INDUSTRY`: 855.720 XNA
- `NAME_CASH_LOAN_PURPOSE`: XAP (922.661) + XNA (677.918)
- `NAME_GOODS_CATEGORY`: 950.809 XNA
- `CODE_REJECT_REASON`: XAP (1.353.093) + XNA (5.244)